[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-09/Tema-18/Tema_18.ipynb)


**Implementación básica con Hugging Face y PyTorch**

## ¿Qué hace este código?

Este ejemplo realiza **clasificación de imágenes _zero-shot_** con **CLIP** (*Contrastive Language–Image Pre-training*) de OpenAI, usando **Hugging Face Transformers** y **PyTorch**. CLIP aprende a relacionar imágenes y texto en un mismo espacio vectorial, por lo que puede clasificar una imagen contra un conjunto de descripciones **sin reentrenar** el modelo.

Paso a paso:

1. **Carga del modelo y el procesador** (`openai/clip-vit-base-patch32`). `CLIPModel` contiene un codificador de imagen (*Vision Transformer*) y uno de texto entrenados para producir vectores comparables; `CLIPProcessor` agrupa el *tokenizer* del texto y el preprocesador de imágenes (redimensionado y normalización). La primera vez se descargan los pesos (~600 MB) desde Hugging Face.
2. **Lectura de la imagen** de entrada con Pillow. En este ejemplo se descarga desde internet una foto de dos gatos del dataset **COCO** (`http://images.cocodataset.org/val2017/000000039769.jpg`), la misma que usa la documentación oficial de CLIP. Si prefieres una imagen local, descomenta la línea `Image.open("imagen.jpg")`.
3. **Etiquetas candidatas** definidas como frases en lenguaje natural (`"un gato"`, `"un perro"`, `"un coche"`, `"una persona"`). Puedes cambiarlas por las clases que quieras reconocer.
4. **Preprocesamiento**: el `processor` tokeniza los textos, redimensiona/normaliza la imagen y devuelve tensores de PyTorch (`return_tensors="pt"`); `padding=True` alinea la longitud de los textos.
5. **Inferencia**: el modelo calcula un *embedding* de la imagen y uno por cada texto, y mide su similitud. `outputs.logits_per_image` contiene una puntuación por etiqueta.
6. **Softmax**: convierte esas puntuaciones en una distribución de **probabilidades** que suman 1, indicando qué tan bien describe cada texto a la imagen.

**Salida esperada:** un tensor con una probabilidad por cada etiqueta de `texts`, en el mismo orden. Para la imagen de ejemplo (dos gatos), la mayor probabilidad corresponde a `"un gato"`, p. ej. `tensor([[0.96, 0.02, 0.01, 0.01]])`.


In [ ]:
# En Google Colab: instala/actualiza transformers (torch, Pillow y requests ya vienen incluidos).
# En un entorno local con las dependencias ya instaladas puedes omitir esta celda.
!pip install -q transformers


In [ ]:
import requests
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Imagen de ejemplo (dos gatos) del dataset COCO, usada en la documentación de CLIP.
# Se descarga desde internet, así no necesitas ningún archivo local.
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Alternativa con imagen local: descomenta la línea siguiente y comenta las 2 anteriores.
# image = Image.open("imagen.jpg")

texts = ["un gato", "un perro", "un coche", "una persona"]

inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)

probs = outputs.logits_per_image.softmax(dim=1)

print(probs)
